# 01d — Extract OSM Pollution Proximity (Overpass API)
**Data source:** [Overpass API](http://overpass-api.de/) — OpenStreetMap features

**Features (per radius: 1km, 5km, 10km):**
- Mines, wastewater plants, farmland, roads within radius
- `osm_mines_Xm`, `osm_wastewater_Xm`, `osm_farmland_Xm`, `osm_roads_Xm`, `osm_total_Xm`

**Total:** 5 features x 3 radii = **15 OSM features** per station

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00

**Output:** `osm.parquet` (one row per unique station)

**Estimated time:** ~30-60 min for ~170 stations (Overpass is slow)

> Enable Internet in Kaggle settings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, time, requests, logging
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01d_osm')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

INPUT_DIR  = '/kaggle/input/ey-water-quality-nb00'
OUTPUT_DIR = '/kaggle/working'

LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

RADII = [1000, 5000, 10000]
log.info(f'Radii: {RADII}')

In [ ]:
train_base = pd.read_parquet(f'{INPUT_DIR}/train_base.parquet')
val_base   = pd.read_parquet(f'{INPUT_DIR}/val_base.parquet')

all_data = pd.concat([train_base, val_base], ignore_index=True)
unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()

log.info(f'Train: {train_base.shape}, Val: {val_base.shape}')
log.info(f'Unique stations: {len(unique_stations)}')

---
## Extraction

In [ ]:
def fetch_osm_detailed(lat, lon, radius_m=5000, retries=3):
    """
    Fetch OSM feature counts near a station.
    Returns separate counts for mines, wastewater, farmland, roads.
    """
    queries = {
        'mines':      f'node(around:{radius_m},{lat},{lon})["man_made"="mine"];',
        'wastewater': f'way(around:{radius_m},{lat},{lon})["man_made"="wastewater_plant"];',
        'farmland':   f'way(around:{radius_m},{lat},{lon})["landuse"="farmland"];',
        'roads':      f'way(around:{radius_m},{lat},{lon})["highway"];',
    }
    
    results = {}
    for name, q in queries.items():
        for attempt in range(retries):
            try:
                query = f'[out:json][timeout:45];({q});out count;'
                r = requests.get('http://overpass-api.de/api/interpreter',
                               params={'data': query}, timeout=60)
                r.raise_for_status()
                count = int(r.json().get('elements', [{}])[0].get('tags', {}).get('total', 0))
                results[f'osm_{name}_{radius_m}m'] = count
                break
            except Exception:
                if attempt < retries - 1:
                    time.sleep(3 * (attempt + 1))  # longer backoff for Overpass
                else:
                    results[f'osm_{name}_{radius_m}m'] = 0
        time.sleep(1.0)  # rate limit between queries
    
    # Total across all categories
    results[f'osm_total_{radius_m}m'] = sum(results.values())
    return results

In [ ]:
total = len(unique_stations)
results = []
start_time = time.time()
errors = 0

log.info(f'Starting OSM extraction: {total} stations x {len(RADII)} radii...')
log.info(f'Expected total API calls: {total * len(RADII) * 4}  (4 categories per radius)')

for idx, row in unique_stations.iterrows():
    lat, lon = row[LAT_COL], row[LON_COL]
    station = row[STATION_COL]
    
    record = {STATION_COL: station, LAT_COL: lat, LON_COL: lon}
    
    for radius in RADII:
        osm_data = fetch_osm_detailed(lat, lon, radius)
        record.update(osm_data)
    
    results.append(record)
    done = len(results)
    
    # Detailed per-station logging
    elapsed = time.time() - start_time
    rate = done / elapsed if elapsed > 0 else 0
    eta = (total - done) / rate if rate > 0 else 0
    
    total_5k = record.get('osm_total_5000m', 0)
    log.info(f'  [{done:3d}/{total}] {done/total*100:5.1f}% | '
             f'ETA {eta/60:.1f}m | {station[:25]} | '
             f'1km={record.get("osm_total_1000m", 0)} '
             f'5km={total_5k} '
             f'10km={record.get("osm_total_10000m", 0)}')

elapsed_total = time.time() - start_time
log.info(f'DONE in {elapsed_total/60:.1f} min')

In [ ]:
osm_df = pd.DataFrame(results)

log.info(f'Output shape: {osm_df.shape}')
log.info(f'Columns: {osm_df.columns.tolist()}')

# Summary per column
osm_cols = [c for c in osm_df.columns if c.startswith('osm_')]
for col in osm_cols:
    log.info(f'  {col:25s}: sum={osm_df[col].sum():6d}, '
             f'max={osm_df[col].max():5d}, '
             f'nonzero={osm_df[col].gt(0).sum()}/{len(osm_df)}')

display(osm_df.describe())

---
## Figure: OSM Feature Maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, radius in enumerate(RADII):
    ax = axes[i]
    col = f'osm_total_{radius}m'
    
    vals = osm_df[col].values
    # Log scale for better visualization
    log_vals = np.log1p(vals)
    
    sc = ax.scatter(osm_df[LON_COL], osm_df[LAT_COL],
                   c=log_vals, cmap='YlOrRd', s=60,
                   edgecolors='gray', linewidths=0.3)
    cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
    cbar.set_label('log(1 + count)', fontsize=9)
    
    n_nonzero = (vals > 0).sum()
    ax.set_title(f'Radius = {radius/1000:.0f} km\n'
                 f'{n_nonzero}/{len(osm_df)} stations with features', fontsize=11)
    ax.set_xlim(16, 33); ax.set_ylim(-35, -22)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.2)

fig.suptitle('OSM Pollution Proximity per Station', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01d_osm_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Category breakdown at 5km
categories = ['mines', 'wastewater', 'farmland', 'roads']
cat_cols = [f'osm_{c}_5000m' for c in categories if f'osm_{c}_5000m' in osm_df.columns]

if cat_cols:
    fig, ax = plt.subplots(figsize=(10, 5))
    totals = [osm_df[c].sum() for c in cat_cols]
    labels = [c.replace('osm_', '').replace('_5000m', '') for c in cat_cols]
    colors_bar = ['#F44336', '#9C27B0', '#4CAF50', '#607D8B']
    
    bars = ax.bar(labels, totals, color=colors_bar[:len(labels)], edgecolor='white')
    ax.bar_label(bars, fontsize=11, fontweight='bold')
    ax.set_title('Total OSM Features Across All Stations (5km radius)')
    ax.set_ylabel('Total count')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_01d_osm_categories.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/osm.parquet'
osm_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024

log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(osm_df)} rows, {len(osm_cols)} features)')
print(f'\n=== DONE ===')
print(f'Output: osm.parquet')
print(f'Rows: {len(osm_df)}, Features: {osm_cols}')
print(f'Next: add this notebook output as dataset input for 01e')